# Track B: reinforcement learning

Train a policy on the model you identified in steps 1 to 3, then fly it on the drone in MATLAB.

1. Write a Gymnasium environment around your model (like `RocketLanderEnv` in TD 4). On the drone, the
   controller measures at each step ($T_s = 0.05$ s): `t`, `z`, `vz`, `az_prev`, `z_ref`, `u_prev`
   (`az_prev`, `u_prev`: acceleration and command of the previous step), and sends $u \in [-1, 1]$.
   `reset()` and `step()` return them in `info["m"]`.
2. Train a Stable-Baselines3 agent with an `MlpPolicy` on it, then export it with
   `export_for_matlab(agent, env)` (`export_policy.py`, do not edit): it writes `policy.mat` and `traj_model.mat`.
   Docs: [Stable-Baselines3](https://stable-baselines3.readthedocs.io/), and to follow the training,
   [TensorBoard with SB3](https://stable-baselines3.readthedocs.io/en/master/guide/tensorboard.html).
3. Write the same mission and observation in `mission()` and `observation()` of `rl_controller.m`, then run
   `EINT_RL_eval` in MATLAB. Complete the TODO in the matlab files to run the mission.

## 1. L'environnement
Il est dans `drone_env.py` (même dossier). La physique est **notre RNN identifié** (`identification_RNN/.../version_0/export_NN`),
recodé en numpy : `h = tanh(Wih*u + Whh*h + bh)`, `az = 20*(Wo*h + bo) - 9.81`, puis intégration de az → vz → z.

- **Action** : u ∈ [-1, 1]
- **Observation** : `[z_ref - z, vz, az_prev/10, u_prev, ∫(z_ref - z)dt]` (recopiée dans `observation()` de `rl_controller.m`)
- **Récompense** : `-|e| - 0.05|vz| - 0.1(u - u_prev)² - 10·max(0, -0.25 - z)` (suivre la consigne, sans à-coups, sans passer sous le sol)
- **Mission** : 0 m jusqu'à 3 s, 0,5 m jusqu'à 7 s, puis -0,25 m (recopiée dans `mission()` de `rl_controller.m`)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from drone_env import DroneEnv, PID, u_hover, run_episode

env = DroneEnv()                                   # mission de la Figure 1, sans bruit
obs, info = env.reset(seed=0)
print("Observation initiale :", obs)
print("Mesures info['m']    :", info["m"])
obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
print("Après un pas         :", obs, " récompense :", round(reward, 3))
print("u d'équilibre (vol stationnaire) du modèle :", round(u_hover(), 3))

## 2. Baseline linéaire : PID
Le sujet demande de comparer le RL à une loi linéaire. Gains réglés par une petite recherche sur grille sur le modèle.

In [ ]:
def plot_runs(runs, title=""):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    for name, L in runs.items():
        ax1.plot(L["t"], L["z"], label=f"{name} (RMSE {L['rmse']:.3f} m, min z {L['z'].min():.2f} m)")
        ax2.step(L["t"], L["u"], where="post", label=name)
    L = next(iter(runs.values()))
    ax1.plot(L["t"], L["z_ref"], "k--", label="consigne")
    ax1.axhline(-0.25, color="gray", ls=":", label="sol")
    ax1.set_ylabel("z [m]"); ax1.legend(); ax1.grid(alpha=0.3); ax1.set_title(title, loc="left")
    ax2.set_ylabel("u"); ax2.set_xlabel("t [s]"); ax2.grid(alpha=0.3); ax2.legend()
    plt.tight_layout(); plt.show()

pid = PID(Kp=0.5, Ki=0.05, Kd=0.45)
L_pid = run_episode(env, pid)
plot_runs({"PID": L_pid}, "Baseline PID sur le modèle identifié")

## 3. Entraînement SAC
On entraîne sur `DroneEnv(randomize=True)` : missions aléatoires (instants et paliers), biais d'accélération aléatoire
(le modèle n'est pas parfait, RMSE ≈ 0,25 m/s²) et un peu de bruit de mesure. Sans ça, l'agent apprend par cœur
**une** trajectoire sur **un** modèle et risque de mal se comporter sur le vrai simulateur.

Durée : ~10-20 min sur CPU pour 100 000 pas. Suivi : `tensorboard --logdir tb_drone` dans un terminal.

In [ ]:
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor

env_train = Monitor(DroneEnv(randomize=True, noise=0.005))

agent = SAC("MlpPolicy", env_train, verbose=1, seed=0,
            learning_starts=5_000,       # pas aléatoires avant d'apprendre
            gamma=0.98,                  # horizon ~ 50 pas = 2,5 s
            tensorboard_log="tb_drone")  # enlever si tensorboard n'est pas installé
agent.learn(total_timesteps=100_000)
agent.save("sac_drone")
# agent = SAC.load("sac_drone")         # pour recharger sans réentraîner

## 4. Évaluation sur la mission de la Figure 1 : RL contre PID

In [ ]:
L_rl = run_episode(env, lambda o: agent.predict(o, deterministic=True)[0])
plot_runs({"SAC": L_rl, "PID": L_pid}, "Mission Figure 1, modèle identifié")

# Robustesse : erreur de modèle (biais sur az) et bruit, sur la même mission
for bias in (-0.5, 0.0, 0.5):
    class Biased(DroneEnv):
        def reset(self, seed=None, options=None):
            out = super().reset(seed=seed); self.az_bias = bias; return out
    e = Biased(noise=0.005)
    r_rl = run_episode(e, lambda o: agent.predict(o, deterministic=True)[0], seed=1)
    r_pid = run_episode(e, pid, seed=1)
    print(f"biais {bias:+.1f} m/s² :  SAC RMSE {r_rl['rmse']:.3f} m (min z {r_rl['z'].min():+.2f})"
          f"   PID RMSE {r_pid['rmse']:.3f} m (min z {r_pid['z'].min():+.2f})")

## 5. Export vers MATLAB
Avec l'environnement **sans bruit ni randomisation** (`env`). Ensuite dans MATLAB : ouvrir `rl_interface/`, lancer `EINT_RL_eval`
(commencer avec `g_noise = 0`, puis 0.1).

In [ ]:
from export_policy import export_for_matlab
export_for_matlab(agent, env)   # écrit policy.mat et traj_model.mat